In [1]:
import glob
import os
from pathlib import Path

import dask.array as da
import napari
import tifffile as tff
import tqdm
import yaml
from multiview_stitcher import fusion
from multiview_stitcher import spatial_image_utils as si_utils

#from napari_autoscape import convert_single_tile_to_ome_zarr, convert_tiles_to_stitched_ome_zarr
from ome_zarr import OMEZarrImage, OMEZarrMultiscale, OMEZarrScene
from ome_zarr_models.v06.coordinate_transforms import (
    Axis,
    CoordinateSystem,
)

c:\Users\aifadmin\Documents\Github\autoSCAPE\.venv\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [2]:
# viewer = napari.Viewer()

# client = Client()  # Dashboard at http://localhost:8787
# print(f"Dashboard: {client.dashboard_link}")

In [3]:
places = [
    r'Z:\Collaborations\aif-botton-collaboration\Johannes\20260826\20260811_BF_W2_full_ImageStackFile_1',
]

## Read tile images

In [ ]:
imgs = []
transforms = []
for place in places:
    tif_files = glob.glob(os.path.join(place, "*.ome.tif"), recursive=True)

    sims = []
    translations = []
    metadata_file = tif_files[0].replace(".ome.tif", "_metadata.txt")
    with open(metadata_file) as f:
        metadata = yaml.safe_load(f)["Summary"]

    for tf in tqdm.tqdm(tif_files):

        tile_index = Path(tf).name.split("MMStack_")[1].replace(".ome.tif", "")

        # index positionlist by label
        positionlist = metadata["InitialPositionList"]
        positionlist = {position["Label"]: position for position in positionlist}
        position = positionlist[tile_index]

        scale = {
            "z": metadata["z-step_um"] if metadata["z-step_um"] != 0 else 1.0,
            "y": metadata["PixelSize_um"],
            "x": metadata["PixelSize_um"]
        }

        array = da.from_array(tff.imread(tf, is_ome=False, is_mmstack=False))
        size_y = array.shape[1] * scale["y"]
        size_x = array.shape[2] * scale["x"]
        size_z = array.shape[0] * scale["z"]

        translation = {
            "z": float(position["DeviceCoordinatesUm"]["ZStage:Z:32"][0]) - size_z // 2,
            "y": - float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][1]) + size_y // 2,
            "x": float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][0]) - size_x // 2,
        }
        translations.append(translation)

        sims.append(
            si_utils.get_sim_from_array(
                array,
                dims=["z", "y", "x"],
                scale=scale,
                translation=translation,
                transform_key="stage_metadata",
                )
        )

    image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion, backend="cupy")

    ngff_img = OMEZarrImage(image.data.squeeze(), axes="zyx", scale=scale,
                            axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"}, name=Path(place).stem)
    ms_img = OMEZarrMultiscale(ngff_img, channel_names=["Brightfield"], channel_colors=["808080"])
    imgs.append(ms_img)

    z_min = min([t["z"] for t in translations])
    y_min = max([t["y"] for t in translations])
    x_min = min([t["x"] for t in translations])

    transforms.append({
        "type": "translation",
        "translation": (z_min, y_min, x_min),
        "input": {"name": "physical", "path": ms_img.name},
        "output": {"name": "world"}
    })

100%|██████████| 361/361 [24:13<00:00,  4.03s/it]


In [17]:
size_y

408.3256

In [16]:
transforms[0]["translation"] = (-53.199999999999996, 2912.0161200000002 - size_y, 2902.18062)
transforms

[{'type': 'translation',
  'translation': (-53.199999999999996, 2503.69052, 2902.18062),
  'input': {'name': 'physical',
   'path': '20260811_BF_W2_full_ImageStackFile_1'},
  'output': {'name': 'world'}}]

In [11]:
world_cs = CoordinateSystem(name="world", axes=[
    Axis(name="z", type="space", unit="micrometer"),
    Axis(name="y", type="space", unit="micrometer"),
    Axis(name="x", type="space", unit="micrometer")])

scene = OMEZarrScene(
    images=imgs,
    coordinate_systems=(world_cs,),
    coordinate_transformations=transforms
)

scene.to_ome_zarr(r"Z:\Collaborations\aif-botton-collaboration\Johannes\20260826\scene.ome.zarr", compute=True, overwrite=True)

Writing images: 100%|██████████| 1/1 [01:00<00:00, 60.37s/it]


[]

## View

In [ ]:
viewer = napari.Viewer()
viewer.open(r"Z:\Collaborations\aif-botton-collaboration\Johannes\20260826\scene.ome.zarr", plugin="napari-ome-zarr")